# 03 Sector Benchmarking
Compare businesses against sector baselines and percentile rankings.

In [1]:
import warnings
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "utils" / "utils.py").exists():
    if (PROJECT_ROOT / "notebooks" / "utils" / "utils.py").exists():
        PROJECT_ROOT = PROJECT_ROOT / "notebooks"
    elif (PROJECT_ROOT.parent / "utils" / "utils.py").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.utils import ensure_project_dirs, load_raw_dataset, clean_dataset, PROCESSED_DIR, REPORTS_DIR, FIGURES_DIR
from utils.features import engineer_kpis, build_post_feature_sets, aggregate_business_features
from utils.evaluation import regression_metrics, rank_models
from utils.visualization import set_plot_style, save_figure

set_plot_style()
ensure_project_dirs()
RAW_DATA_PATH = Path(r"C:\univ\Data mining\Project\synthetic_generator\synthetic_social_media_posts.csv")
if not RAW_DATA_PATH.exists():
    RAW_DATA_PATH = PROJECT_ROOT / "synthetic_generator" / "synthetic_social_media_posts.csv"
KPI_PATH = PROCESSED_DIR / "kpi_dataset.csv"

## Load KPI Dataset

In [2]:
if KPI_PATH.exists():
    df = pd.read_csv(KPI_PATH, parse_dates=["post_date"])
else:
    df = engineer_kpis(clean_dataset(load_raw_dataset(RAW_DATA_PATH)))
df.head()

,business_name,sector,followers_count,post_date,posting_hour,day_of_week,month,post_type,caption_text,caption_length,...,is_reel,posting_time_bin,caption_length_bin,hashtags_count_bin,emoji_count_bin,engagement_level,business_size_bin,high_engagement,high_view_rate,high_comment_rate
0,Ramallah Care Pharmacy,pharmacy,13666,2025-12-16,22,Tuesday,12,carousel,صحتك بتهمنا Swipe وشوفوا الباقي. موجودين في Ra...,109,...,0,night,medium,few,none,low,large,0,0,0
1,Nablus Bites,restaurant,12838,2025-10-18,20,Saturday,10,video,اليوم عنا أكل طيب فيديو جديد. احنا جاهزين احجز...,78,...,0,evening,medium,few,none,medium,medium,0,0,0
2,Al Amal Dental,clinic,4652,2025-01-17,21,Friday,1,carousel,صحتكم أولويتنا سوايب وشوفوا التفاصيل. شو رأيكم...,142,...,0,night,long,few,few,low,small,0,0,0
3,Hebron Study Hub,education_center,26289,2025-03-27,20,Thursday,3,reel,استنونا بدورة جديدة شوفوا الريل. زورونا بفرعنا...,90,...,1,evening,medium,few,few,high,large,1,1,1
4,Bethlehem Brew Bar,cafe,2210,2025-11-01,22,Saturday,11,image,أجواء ولا أروع صورة جديدة. أهلا بالناس الحلوة ...,72,...,0,night,medium,few,none,medium,small,0,0,0


## Build and Compare Ranking Methods

In [3]:
biz = df.groupby(["business_name","sector"], as_index=False).agg(
    followers_count=("followers_count","median"),
    posts_count=("business_name","size"),
    engagement=("engagement","mean"),
    engagement_rate=("engagement_rate","mean"),
    view_rate=("view_rate","mean"),
    comment_rate=("comment_rate","mean"),
)
biz["score_engagement_only"] = biz["engagement_rate"]
biz["score_balanced"] = 0.5*biz["engagement_rate"] + 0.3*biz["view_rate"] + 0.2*biz["comment_rate"]
biz["score_sector_z"] = 0.0
for sec, idx in biz.groupby("sector").groups.items():
    seg = biz.loc[idx, ["engagement_rate","view_rate","comment_rate"]]
    z = (seg - seg.mean()) / seg.std(ddof=0).replace(0,1)
    biz.loc[idx, "score_sector_z"] = (0.5*z["engagement_rate"] + 0.3*z["view_rate"] + 0.2*z["comment_rate"]).values

long = []
for m in ["score_engagement_only","score_balanced","score_sector_z"]:
    tmp = biz[["business_name","sector",m]].copy()
    tmp["percentile"] = tmp.groupby("sector")[m].rank(pct=True)*100
    tmp["method"] = m
    long.append(tmp)
long = pd.concat(long, ignore_index=True)
method_eval = rank_models(
    long.groupby("method", as_index=False).agg(avg_percentile=("percentile","mean"), percentile_std=("percentile","std")),
    higher_is_better_cols=["avg_percentile"], lower_is_better_cols=["percentile_std"]
)
best_method = "score_balanced"

## Produce Benchmark Outputs and Save

In [4]:
sector_avg = df.groupby("sector", as_index=False).agg(
    sector_avg_engagement=("engagement","mean"),
    sector_avg_engagement_rate=("engagement_rate","mean"),
)
bench = biz.copy()
bench["benchmark_score"] = bench[best_method]
bench["business_percentile_rank"] = bench.groupby("sector")["benchmark_score"].rank(pct=True)*100
best_post_types = df.groupby(["sector","post_type"], as_index=False)["engagement_rate"].mean().sort_values(["sector","engagement_rate"], ascending=[True,False]).groupby("sector", as_index=False).first().rename(columns={"post_type":"best_post_type","engagement_rate":"best_post_type_engagement_rate"})
top_businesses = bench.sort_values(["sector","business_percentile_rank"], ascending=[True,False]).groupby("sector", as_index=False).head(3)
sector_benchmarking = bench.merge(sector_avg, on="sector", how="left").merge(best_post_types, on="sector", how="left")

sector_benchmarking.to_csv(PROCESSED_DIR / "sector_benchmarking.csv", index=False)
method_eval.to_csv(REPORTS_DIR / "sector_benchmarking_methods.csv", index=False)
top_businesses.to_csv(REPORTS_DIR / "top_businesses_by_sector.csv", index=False)
display(sector_benchmarking.head(15))
print("Insight: balanced score selected for better trade-off across engagement, reach, and comments.")

,business_name,sector,followers_count,posts_count,engagement,engagement_rate,view_rate,comment_rate,score_engagement_only,score_balanced,score_sector_z,benchmark_score,business_percentile_rank,sector_avg_engagement,sector_avg_engagement_rate,best_post_type,best_post_type_engagement_rate
0,Al Amal Dental,clinic,4681.0,26,515.461538,0.110404,1.381762,0.008868,0.110404,0.471504,1.126921,0.471504,100.0,1143.754902,0.093695,reel,0.145082
1,Al Balad Grill,restaurant,7518.0,27,1241.555556,0.165165,1.456613,0.012499,0.165165,0.522066,0.570265,0.522066,100.0,1272.370000,0.156801,reel,0.242216
2,Al Hayat Pharmacy,pharmacy,6472.0,31,652.064516,0.100502,1.269667,0.008197,0.100502,0.432791,0.989887,0.432791,75.0,812.009091,0.085343,reel,0.162331
3,Al Karmel Boutique,store,9211.0,27,880.481481,0.094904,0.773967,0.009104,0.094904,0.281463,-1.383849,0.281463,25.0,4164.427419,0.140456,reel,0.251457
4,Bethlehem Brew Bar,cafe,2230.0,27,436.296296,0.195922,1.460202,0.015227,0.195922,0.539067,-0.626395,0.539067,25.0,748.872340,0.200186,reel,0.352729
5,Bethlehem Lash Bar,beauty_salon,13437.0,22,1587.181818,0.118170,1.134320,0.010409,0.118170,0.401463,-0.681392,0.401463,50.0,667.815789,0.136087,reel,0.235985
6,Canaan Threads,store,14897.5,34,2152.147059,0.144083,1.171074,0.013058,0.144083,0.425975,0.180949,0.425975,75.0,4164.427419,0.140456,reel,0.251457
7,Gaza Family Pharmacy,pharmacy,12513.0,29,836.862069,0.066539,1.015314,0.004719,0.066539,0.338808,-1.131622,0.338808,25.0,812.009091,0.085343,reel,0.162331
8,Gaza Gadget Shop,electronics_store,49419.0,37,5507.783784,0.112031,1.008816,0.008052,0.112031,0.360271,-0.381253,0.360271,50.0,5263.251969,0.120778,reel,0.235001
9,Gaza Glow Salon,beauty_salon,1171.0,18,135.777778,0.116361,0.899981,0.008201,0.116361,0.329815,-1.101465,0.329815,25.0,667.815789,0.136087,reel,0.235985


Insight: balanced score selected for better trade-off across engagement, reach, and comments.
